# Notebook 12 — Final Model Selection

**Hyperparameter-optimisation experiment · Soccer Event Transformer · Track A model family**

| Item | Content |
|---|---|
| **Question** | Which configuration should be trained as the final model of this experiment, and does it replace the frozen Track A model? |
| **Purpose of this notebook** | Apply a selection rule, stated before any result is read, to the five-seed validation results; record the decision and its evidence; answer the replacement question separately; write the frozen configuration that final training uses. |
| **Inputs** | Validation results only: the capacity sweep (`results/08_capacity_*.csv`), the candidate list (`results/09_candidates.json`), the five-seed benchmark (`results/11_seed_runs.csv`, `11_seed_summary.csv`, `11_pairwise.csv`, `11_best_versus_candidates.csv`, `11_stability_report.json`) and, if present, the saturation report (`results/10_elbow_report.json`). |
| **Training performed** | None. This notebook only reads result files. |
| **Test data** | Not used and not accessible. The test split can only be unlocked once a frozen configuration exists; this notebook writes it. |
| **Outputs** | `config/frozen_config.json`, `results/12_selected_config.json`, `results/12_selection_evidence.csv`, `results/12_summary.json`. |

Authored by: Alisha Shaikh (PICT, Pune)

In [ ]:
# ---------------------------------------------------------------------------
# Environment setup (identical at the top of every notebook of this experiment)
# ---------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import copy
import hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_DIR = Path(os.environ.get("TRACK_A_DIR", "/content/drive/MyDrive/soccer_mi/track_a"))
RUN_MODE = os.environ.get("TRACK_A_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Existing Track A outputs.
EXISTING_DIR = PROJECT_DIR if RUN_MODE == "full" else PROJECT_DIR / "smoke"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
EXISTING_RUNS_DIR = EXISTING_DIR / "runs"
EXISTING_RESULTS_DIR = EXISTING_DIR / "results"
EXISTING_FROZEN_CONFIG = EXISTING_DIR / "config" / "frozen_config.json"

OPT_DIR = EXISTING_DIR / "experiments" / "hyperparameter_optimisation"
RUNS_DIR = OPT_DIR / "runs"
RESULTS_DIR = OPT_DIR / "results"
FIGURES_DIR = OPT_DIR / "figures"
CONFIG_DIR = OPT_DIR / "config"
MANIFEST_DIR = OPT_DIR / "manifests"
LOG_DIR = OPT_DIR / "logs"
for folder in [RUNS_DIR, RESULTS_DIR, FIGURES_DIR, CONFIG_DIR, MANIFEST_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = MANIFEST_DIR / "experiment_manifest.jsonl"


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("Track A project (read-only):", EXISTING_DIR)
print("Experiment outputs:", OPT_DIR)

Mounted at /content/drive
Colab: True | RUN_MODE: full
Track A project (read-only): /content/drive/MyDrive/soccer_mi/track_a
Experiment outputs: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import importlib

import src.data as data
import src.batching as batching
import src.metrics as metrics
import src.model as model_module
import src.mlp_baseline as mlp_baseline
import src.training as training
import src.analysis as analysis
for module in [data, batching, metrics, model_module, mlp_baseline, training, analysis]:
    importlib.reload(module)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if RUN_MODE == "full" and DEVICE != "cuda":
    raise RuntimeError("BLOCKED: full-mode training requires a GPU. In Colab: Runtime > Change runtime type > GPU.")

events = pd.read_parquet(ARTIFACTS_DIR / "events_prepared.parquet")
targets = events["action_id"].to_numpy().astype("int64")
match_ids = events["match_id"].to_numpy()

with open(EXISTING_FROZEN_CONFIG) as handle:
    TRACK_A_FROZEN = json.load(handle)

ENCODING = TRACK_A_FROZEN["encoding"]
POSSESSION_FEATURES = TRACK_A_FROZEN["possession_features"]
STRIDE = TRACK_A_FROZEN["stride"]

BASE_LAYERS = TRACK_A_FROZEN["model"]["n_layers"]
BASE_HEADS = TRACK_A_FROZEN["model"]["n_heads"]
BASE_D_MODEL = TRACK_A_FROZEN["model"]["d_model"]
BASE_D_FF = TRACK_A_FROZEN["model"]["d_ff"]
BASE_CONTEXT = TRACK_A_FROZEN["context_length"]
BASE_LR = TRACK_A_FROZEN["training"]["learning_rate"]
BASE_WD = TRACK_A_FROZEN["training"]["weight_decay"]
EPOCH_BUDGET = TRACK_A_FROZEN["training"]["max_epochs"]

print("Events:", len(events), "| device:", DEVICE)
print("Track A reference configuration:",
      f"L{BASE_LAYERS} H{BASE_HEADS} d_model {BASE_D_MODEL} d_ff {BASE_D_FF} K{BASE_CONTEXT}",
      f"| lr {BASE_LR:g} wd {BASE_WD:g} | epoch budget {EPOCH_BUDGET}")

Events: 658905 | device: cuda
Track A reference configuration: L2 H2 d_model 128 d_ff 512 K16 | lr 0.0003 wd 0.1 | epoch budget 30


In [ ]:
SHORT_NAME = {"batch_size": "bs", "warmup_fraction": "wu"}


def transformer_config(seed, n_layers, n_heads, context_length, d_model=None, d_ff=None,
                       learning_rate=None, weight_decay=None, max_epochs=None):
    """Build a run configuration in exactly the format used by the Track A notebooks."""
    model = {"n_layers": n_layers, "n_heads": n_heads,
             "d_model": BASE_D_MODEL if d_model is None else d_model,
             "d_ff": BASE_D_FF if d_ff is None else d_ff}
    training_settings = {"learning_rate": BASE_LR if learning_rate is None else learning_rate,
                         "weight_decay": BASE_WD if weight_decay is None else weight_decay,
                         "max_epochs": EPOCH_BUDGET if max_epochs is None else max_epochs}
    return {"model_type": "transformer", "encoding": ENCODING, "possession_features": POSSESSION_FEATURES,
            "context_length": context_length, "stride": STRIDE, "seed": seed,
            "model": model, "training": training_settings}


def architecture_label(config):
    """Readable architecture name, e.g. L4-H2-K16-D128-F512."""
    model = config["model"]
    return (f"L{model['n_layers']}-H{model['n_heads']}-K{config['context_length']}"
            f"-D{model['d_model']}-F{model['d_ff']}")


def variant_tag(training_settings):
    """Sub-folder for training settings that training.run_name_for() does not encode.

    run_name_for() encodes the learning rate, weight decay, architecture, context length and
    seed, but not the batch size or the warm-up fraction. To prevent runs that differ only in those
    two settings (batch size or the warm-up fraction) to be written to the same folder.
    """
    parts = []
    for key in ["batch_size", "warmup_fraction"]:
        if key in training_settings and training_settings[key] != training.DEFAULT_TRAINING[key]:
            parts.append(f"{SHORT_NAME[key]}{training_settings[key]:g}")
    return "_".join(parts) if parts else "base"


_BUNDLES = {}


def bundle_for(context_length):
    """Encoded data and evaluation views for one context length (built once, then reused)."""
    if context_length not in _BUNDLES:
        _BUNDLES[context_length] = training.prepare_data(
            events, ARTIFACTS_DIR, ENCODING, POSSESSION_FEATURES, context_length, STRIDE, DEVICE, RUN_MODE)
    return _BUNDLES[context_length]


def parameter_count(config):
    """Total parameters of a configuration, without training it."""
    model = training.build_model(config, bundle_for(config["context_length"])["vocabulary_sizes"])
    return model_module.count_parameters(model)["total"]


def run_one(config, extra_training=None, keep_every_epoch=False, verbose=True):
    """Train one configuration (or reload it if its folder already holds a finished run).

    Returns (run_dir, final_results). Nothing outside RUNS_DIR is written.
    """
    config = copy.deepcopy(config)
    if extra_training:
        config["training"].update(extra_training)
    runs_root = RUNS_DIR / variant_tag(config["training"])
    runs_root.mkdir(parents=True, exist_ok=True)
    result = training.train_run(config, bundle_for(config["context_length"]), runs_root,
                                MANIFEST_PATH, keep_every_epoch=keep_every_epoch, verbose=verbose)
    return runs_root / training.run_name_for(config), result


def result_row(config, run_dir, result, group=""):
    """One flat record per run, holding every quantity the later notebooks need."""
    val = result["val_metrics"]
    training_settings = result["training_settings"]
    return {
        "group": group,
        "architecture": architecture_label(config),
        "n_layers": config["model"]["n_layers"],
        "n_heads": config["model"]["n_heads"],
        "d_head": config["model"]["d_model"] // config["model"]["n_heads"],
        "d_model": config["model"]["d_model"],
        "d_ff": config["model"]["d_ff"],
        "context_length": config["context_length"],
        "learning_rate": training_settings["learning_rate"],
        "weight_decay": training_settings["weight_decay"],
        "batch_size": training_settings["batch_size"],
        "warmup_fraction": training_settings["warmup_fraction"],
        "seed": config["seed"],
        "parameters": result["parameter_counts"]["total"],
        "trainable_parameters": result["parameter_counts"].get("trainable", result["parameter_counts"]["total"]),
        "val_nll": val["nll"],
        "val_accuracy": val["accuracy"],
        "val_macro_f1": val["macro_f1"],
        "val_top3": val["top3_accuracy"],
        "val_top5": val["top5_accuracy"],
        "val_ece": val["ece"],
        "train_nll": result["train_nll"],
        "best_epoch": result["best_epoch"],
        "epochs_run": result["epochs_run"],
        "stopped_early": result["stopped_early"],
        "diverged": result["diverged"],
        "runtime_seconds": result["runtime_seconds"],
        "run_dir": str(run_dir),
        "run_name": result["run_name"],
    }


def append_results(rows, csv_path):
    """Append run records to a CSV, keeping one row per run name and variant folder."""
    table = pd.DataFrame(rows)
    if Path(csv_path).exists():
        table = pd.concat([pd.read_csv(csv_path), table], ignore_index=True)
    table = table.drop_duplicates(subset=["run_dir"], keep="last").reset_index(drop=True)
    table.to_csv(csv_path, index=False)
    return table


def seed_level_to_config_level(table):
    """Average the seed-level records of each configuration into one row per configuration."""
    keys = ["group", "architecture", "n_layers", "n_heads", "d_head", "d_model", "d_ff",
            "context_length", "learning_rate", "weight_decay", "batch_size", "warmup_fraction",
            "parameters"]
    summary = table.groupby(keys, as_index=False, sort=False).agg(
        n_seeds=("seed", "count"),
        val_nll=("val_nll", "mean"),
        val_nll_sd=("val_nll", "std"),
        val_accuracy=("val_accuracy", "mean"),
        val_accuracy_sd=("val_accuracy", "std"),
        val_macro_f1=("val_macro_f1", "mean"),
        val_top3=("val_top3", "mean"),
        val_top5=("val_top5", "mean"),
        val_ece=("val_ece", "mean"),
        train_nll=("train_nll", "mean"),
        best_epoch=("best_epoch", "mean"),
        runtime_seconds=("runtime_seconds", "mean"),
    )
    return summary.sort_values("val_nll").reset_index(drop=True)


def run_dirs_for(table, **conditions):
    """All run directories in a results table that match the given column values."""
    selected = table
    for column, value in conditions.items():
        selected = selected[selected[column] == value]
    return [Path(path) for path in selected["run_dir"]]


print("Helper functions defined.")

Helper functions defined.


## 1. The selection rule

**Candidate set.** The benchmark candidates: six architectures (8L_2H, 6L_1H, 6L_2H, 6L_8H, 4L_2H and 2L_2H, all K = 16, d_model 128, d_ff 512), each trained with the tuned setting (learning rate 3e-4, weight decay 0.01) and seeds 10–14. They differ in architecture only. The frozen Track A reference (weight decay 0.1) and the tuning check (6L_2H at weight decay 0.1) are reported but are **not eligible**, because they use different training hyperparameters.

**Primary metric.** Mean validation NLL over the five seeds. NLL scores the whole predicted distribution rather than only the top-ranked class, and it is the metric under which the Track A model was selected.

**Secondary metrics.** Accuracy, macro-F1, top-3, top-5 and ECE are reported for the selected configuration.

**Decision rule ("better beyond noise").** A is better than B beyond noise only if both hold: (1) the 95% match-level bootstrap interval of the per-event NLL difference (A − B), computed on per-event validation NLL averaged over seeds, lies entirely below zero; and (2) every seed of A has a lower validation NLL than every seed of B. This is the rule under which the Track A model was selected.

**Step 1 — best candidate.** The benchmark candidate with the lowest mean validation NLL.

**Step 2 — smallest candidate within noise.** Order the benchmark candidates from smallest to largest by parameter count; break ties by head-position units (layers × heads × K), then by mean validation NLL. The selected configuration is the first one that the best candidate does **not** beat beyond noise.

**Step 3 — selected configuration.** The result of step 2.

**Step 4 — replacement decision, kept separate.** A replacement is proposed only if the selected configuration beats the Track A reference beyond noise under the same rule. Otherwise the Track A model is retained and the selected configuration


## 2. Evidence

The table lists all eight candidates with their five-seed validation results. Only rows with role `benchmark` enter steps 1–3.

In [ ]:
required_files = {
    "capacity runs": RESULTS_DIR / "08_capacity_runs.csv",
    "capacity summary": RESULTS_DIR / "08_capacity_summary.csv",
    "candidates": RESULTS_DIR / "09_candidates.json",
    "seed runs": RESULTS_DIR / "11_seed_runs.csv",
    "seed summary": RESULTS_DIR / "11_seed_summary.csv",
    "pairwise comparisons": RESULTS_DIR / "11_pairwise.csv",
    "best versus candidates": RESULTS_DIR / "11_best_versus_candidates.csv",
    "stability report": RESULTS_DIR / "11_stability_report.json",
}
missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise RuntimeError(f"BLOCKED: missing inputs from earlier notebooks: {missing}")

seed_table = pd.read_csv(required_files["seed runs"])
stability_table = pd.read_csv(required_files["seed summary"])
pairwise = pd.read_csv(required_files["pairwise comparisons"])
best_versus = pd.read_csv(required_files["best versus candidates"])
with open(required_files["stability report"]) as handle:
    stability_report = json.load(handle)
with open(required_files["candidates"]) as handle:
    CANDIDATES = json.load(handle)["candidates"]

BENCHMARK_TABLE = stability_table[stability_table["role"] == "benchmark"].reset_index(drop=True)

elbow_path = RESULTS_DIR / "10_elbow_report.json"
elbow_report = json.load(open(elbow_path)) if elbow_path.exists() else None

candidate_run_dirs = {}
for name in stability_table["candidate"]:
    candidate_run_dirs[name] = [Path(path) for path in
                                seed_table[seed_table["candidate"] == name]["run_dir"]]

print("Candidates and their validation results over the stability seeds:")
stability_table[["candidate", "role", "architecture", "parameters", "learning_rate", "weight_decay", "n_seeds",
                 "val_nll_mean", "val_nll_sd", "val_accuracy_mean", "val_macro_f1_mean",
                 "val_top3_mean", "val_top5_mean", "val_ece_mean"]].round(4)

Candidates and their validation results over the stability seeds:


,candidate,role,architecture,parameters,learning_rate,weight_decay,n_seeds,val_nll_mean,val_nll_sd,val_accuracy_mean,val_macro_f1_mean,val_top3_mean,val_top5_mean,val_ece_mean
0,8L_2H_K16,benchmark,L8-H2-K16-D128-F512,1704207,0.0003,0.01,5,0.9010,0.0005,0.6218,0.5618,0.9482,0.9946,0.0167
1,6L_1H_K16,benchmark,L6-H1-K16-D128-F512,1307663,0.0003,0.01,5,0.9021,0.0015,0.6218,0.5577,0.9483,0.9944,0.0170
2,6L_2H_K16,benchmark,L6-H2-K16-D128-F512,1307663,0.0003,0.01,5,0.9025,0.0015,0.6223,0.5601,0.9485,0.9945,0.0164
3,6L_2H_K16 (Track A hyperparameters),tuning check,L6-H2-K16-D128-F512,1307663,0.0003,0.10,5,0.9031,0.0010,0.6222,0.5593,0.9480,0.9944,0.0230
4,4L_2H_K16,benchmark,L4-H2-K16-D128-F512,911119,0.0003,0.01,5,0.9036,0.0017,0.6216,0.5590,0.9482,0.9944,0.0196
5,6L_8H_K16,benchmark,L6-H8-K16-D128-F512,1307663,0.0003,0.01,5,0.9043,0.0017,0.6215,0.5600,0.9480,0.9944,0.0199
6,Track A reference,reference,L2-H2-K16-D128-F512,514575,0.0003,0.10,5,0.9069,0.0018,0.6208,0.5583,0.9474,0.9941,0.0161
7,2L_2H_K16,benchmark,L2-H2-K16-D128-F512,514575,0.0003,0.01,5,0.9070,0.0020,0.6208,0.5583,0.9474,0.9941,0.0164


## 3. Step 1 — the best candidate by validation NLL

**RESULT.** The lowest mean validation NLL among the benchmark candidates belongs to **8L_2H_K16** (L8-H2-K16-D128-F512, 1,704,207 parameters): 0.9010 (seed SD 0.0005), accuracy 0.6218, macro-F1 0.5618, ECE 0.0167.

Full ordering: 8L_2H_K16 0.9010, 6L_1H_K16 0.9021, 6L_2H_K16 0.9025, 4L_2H_K16 0.9036, 6L_8H_K16 0.9043, 2L_2H_K16 0.9070.

In [ ]:
ordered = BENCHMARK_TABLE.sort_values("val_nll_mean").reset_index(drop=True)
BEST_CANDIDATE = str(ordered.iloc[0]["candidate"])
print(f"Lowest mean validation NLL among the benchmark candidates: {BEST_CANDIDATE}")
print(f"  architecture     {ordered.iloc[0]['architecture']}")
print(f"  parameters       {int(ordered.iloc[0]['parameters']):,}")
print(f"  mean val NLL     {ordered.iloc[0]['val_nll_mean']:.4f} (SD {ordered.iloc[0]['val_nll_sd']:.4f})")
print(f"  mean val acc     {ordered.iloc[0]['val_accuracy_mean']:.4f}")
print(f"  mean macro-F1    {ordered.iloc[0]['val_macro_f1_mean']:.4f}")
print(f"  mean ECE         {ordered.iloc[0]['val_ece_mean']:.4f}")
print()
print("Full ordering of the benchmark candidates by mean validation NLL:")
print(ordered[["candidate", "architecture", "parameters", "val_nll_mean", "val_nll_sd",
               "val_accuracy_mean"]].round(4).to_string(index=False))

Lowest mean validation NLL among the benchmark candidates: 8L_2H_K16
  architecture     L8-H2-K16-D128-F512
  parameters       1,704,207
  mean val NLL     0.9010 (SD 0.0005)
  mean val acc     0.6218
  mean macro-F1    0.5618
  mean ECE         0.0167

Full ordering of the benchmark candidates by mean validation NLL:
candidate        architecture  parameters  val_nll_mean  val_nll_sd  val_accuracy_mean
8L_2H_K16 L8-H2-K16-D128-F512     1704207        0.9010      0.0005             0.6218
6L_1H_K16 L6-H1-K16-D128-F512     1307663        0.9021      0.0015             0.6218
6L_2H_K16 L6-H2-K16-D128-F512     1307663        0.9025      0.0015             0.6223
4L_2H_K16 L4-H2-K16-D128-F512      911119        0.9036      0.0017             0.6216
6L_8H_K16 L6-H8-K16-D128-F512     1307663        0.9043      0.0017             0.6215
2L_2H_K16 L2-H2-K16-D128-F512      514575        0.9070      0.0020             0.6208


## 4. Step 2 — the smallest candidate within noise of the best

The benchmark candidates are ordered from smallest to largest (parameter count, then head-position units, then mean validation NLL), and each is compared with the best candidate under the decision rule. The selected configuration is the first one the best candidate does not beat beyond noise.


In [ ]:
size_order = BENCHMARK_TABLE.copy()
size_order["head-position units"] = size_order["n_layers"] * size_order["n_heads"] * size_order["context_length"]
size_order = size_order.sort_values(["parameters", "head-position units", "val_nll_mean"])
by_size = [(row["candidate"], candidate_run_dirs[row["candidate"]]) for _, row in size_order.iterrows()]
SELECTED_CANDIDATE, selection_evidence = analysis.smallest_within_noise(by_size, targets, match_ids)
selection_evidence.to_csv(RESULTS_DIR / "12_selection_evidence.csv")
if SELECTED_CANDIDATE != stability_report["smallest_within_noise_of_best"]:
    print(f"Note: notebook 11 recorded {stability_report['smallest_within_noise_of_best']}; the recomputation here "
          f"gives {SELECTED_CANDIDATE}. The value computed here is used.")

print(f"Selected configuration: {SELECTED_CANDIDATE}")
if SELECTED_CANDIDATE != BEST_CANDIDATE:
    print(f"This is not the candidate with the lowest mean NLL ({BEST_CANDIDATE}). "
          f"The lowest-NLL candidate does not beat it beyond noise, and the smaller configuration "
          f"is preferred under step 2 of the rule.")
else:
    print("The candidate with the lowest mean NLL is also the smallest one within noise of the best.")
print()
print(selection_evidence.to_string())

Selected configuration: 4L_2H_K16
This is not the candidate with the lowest mean NLL (8L_2H_K16). The lowest-NLL candidate does not beat it beyond noise, and the smaller configuration is preferred under step 2 of the rule.

           mean val NLL  is best  best beats it beyond noise       best - candidate (CI) seeds separated
candidate                                                                                               
2L_2H_K16        0.9070    False                        True  -0.0060 [-0.0080, -0.0041]            True
4L_2H_K16        0.9036    False                       False  -0.0026 [-0.0040, -0.0013]           False
6L_1H_K16        0.9021    False                       False  -0.0011 [-0.0027, +0.0004]           False
6L_2H_K16        0.9025    False                       False  -0.0015 [-0.0027, -0.0003]           False
6L_8H_K16        0.9043    False                        True  -0.0033 [-0.0048, -0.0018]            True
8L_2H_K16        0.9010     True         

**What the table shows.** One row per benchmark candidate, smallest first. `mean val NLL` is the mean over seeds 10–14; `is best` marks the lowest; `best beats it beyond noise` records whether the best candidate satisfies both parts of the decision rule against that row; the interval column gives the difference (best − candidate) with its 95% match-level bootstrap interval; `seeds separated` is part 2 of the rule.

**RESULT.**

* 2L_2H_K16 (514,575 parameters): the best candidate beats it beyond noise (−0.0060 [−0.0080, −0.0041], seeds separated).
* **4L_2H_K16 (911,119 parameters): not beaten beyond noise** (−0.0026 [−0.0040, −0.0013], seeds not separated). This is the first such row, so it is the selected configuration.
* The larger rows (6L_1H, 6L_2H, 6L_8H) are not needed for the selection; 6L_8H_K16 is beaten beyond noise, 6L_1H_K16 and 6L_2H_K16 are not.

**INTERPRETATION.** 4L_2H_K16 is selected although it is not the candidate with the lowest mean NLL. Its bootstrap interval against 8L_2H_K16 lies below zero, so the 8-layer model has the lower NLL on these validation matches, but the seed results of the two overlap, so the difference does not satisfy the decision rule. Under step 2 the smaller configuration is preferred.

**What we cannot conclude.** "Not beaten beyond noise" is not "equal". The data are consistent with the 8-layer model being slightly better; the rule only says that this has not been established across both matches and seeds.

## 5. Secondary metrics of the selected configuration

The secondary metrics do not enter the selection. They are checked so that a configuration that wins on NLL while being clearly worse on accuracy, macro-F1 or calibration would be reported as such. For each metric the table gives the selected configuration's value, the best value among the six benchmark candidates, and the selected configuration's rank (1 = best).

In [ ]:
selected_row = stability_table[stability_table["candidate"] == SELECTED_CANDIDATE].iloc[0]
metric_columns = ["val_nll_mean", "val_accuracy_mean", "val_macro_f1_mean", "val_top3_mean",
                  "val_top5_mean", "val_ece_mean"]
lower_is_better = {"val_nll_mean", "val_ece_mean"}

secondary_rows = []
for column in metric_columns:
    values = BENCHMARK_TABLE[column]
    if column in lower_is_better:
        rank = int((values < selected_row[column]).sum()) + 1
        best_value = values.min()
    else:
        rank = int((values > selected_row[column]).sum()) + 1
        best_value = values.max()
    secondary_rows.append({"metric": column.replace("val_", "").replace("_mean", ""),
                           "selected value": selected_row[column],
                           "best value among benchmark candidates": best_value,
                           "rank of selected (1 = best)": rank,
                           "lower is better": column in lower_is_better})
secondary = pd.DataFrame(secondary_rows)
print(secondary.round(4).to_string(index=False))

disagreements = secondary[secondary["rank of selected (1 = best)"] > 1]
if len(disagreements):
    print("\nMetrics on which another benchmark candidate ranks higher than the selected configuration:")
    print(disagreements[["metric", "selected value", "best value among benchmark candidates"]].round(4).to_string(index=False))
    print("These are reported, not acted on: the selection rule uses NLL alone.")
else:
    print("\nThe selected configuration also ranks first on every secondary metric.")

  metric  selected value  best value among benchmark candidates  rank of selected (1 = best)  lower is better
     nll          0.9036                                 0.9010                            4             True
accuracy          0.6216                                 0.6223                            4            False
macro_f1          0.5590                                 0.5618                            4            False
    top3          0.9482                                 0.9485                            4            False
    top5          0.9944                                 0.9946                            4            False
     ece          0.0196                                 0.0164                            5             True

Metrics on which another benchmark candidate ranks higher than the selected configuration:
  metric  selected value  best value among benchmark candidates
     nll          0.9036                                 0.9010
accuracy  

**RESULT.** 4L_2H_K16 ranks 4th of 6 on NLL (0.9036 against 0.9010), accuracy (0.6216 against 0.6223), macro-F1 (0.5590 against 0.5618), top-3 (0.9482 against 0.9485) and top-5 (0.9944 against 0.9946), and 5th on ECE (0.0196 against 0.0164).

**INTERPRETATION.** The secondary metrics agree with NLL: the selected configuration sits in the middle of the benchmark on every metric. The gaps to the best values are small (accuracy 0.6216 against 0.6223, top-5 0.9944 against 0.9946), and it is not clearly worse on any secondary metric than on NLL. These differences are reported, not acted on; the rule uses NLL alone.

## 6. Step 4 — replacement decision


In [ ]:
TRACK_A_CANDIDATE = "Track A reference"
if TRACK_A_CANDIDATE not in candidate_run_dirs:
    raise RuntimeError("BLOCKED: the Track A reference is not among the candidates.")

if SELECTED_CANDIDATE == TRACK_A_CANDIDATE:
    replacement_comparison = None
    REPLACES_TRACK_A = False
    replacement_statement = ("The selected configuration is the Track A reference itself. "
                             "No replacement arises: the experiment found no configuration that "
                             "improves on the current model beyond noise.")
else:
    replacement_comparison = analysis.compare_conditions(
        candidate_run_dirs[SELECTED_CANDIDATE], candidate_run_dirs[TRACK_A_CANDIDATE],
        targets, match_ids)
    REPLACES_TRACK_A = bool(replacement_comparison["a_better_beyond_noise"])
    if REPLACES_TRACK_A:
        replacement_statement = (
            f"The selected configuration beats the Track A reference beyond noise on validation "
            f"({replacement_comparison['difference_a_minus_b']:+.4f} nats, 95% interval "
            f"[{replacement_comparison['difference_ci_low']:+.4f}, "
            f"{replacement_comparison['difference_ci_high']:+.4f}], seeds separated). "
            f"A replacement is proposed, subject to the test-set result in notebook 13.")
    else:
        replacement_statement = (
            f"The selected configuration does not beat the Track A reference beyond noise "
            f"({replacement_comparison['difference_a_minus_b']:+.4f} nats, 95% interval "
            f"[{replacement_comparison['difference_ci_low']:+.4f}, "
            f"{replacement_comparison['difference_ci_high']:+.4f}], seeds separated: "
            f"{replacement_comparison['seeds_completely_separated']}). "
            f"The Track A model is retained. The selected configuration is reported as the best "
            f"configuration found in this experiment, not as a replacement.")

print("Best configuration found in this experiment:", SELECTED_CANDIDATE)
print("Proposed as a replacement for the frozen Track A model:", REPLACES_TRACK_A)
print()
print(replacement_statement)

Best configuration found in this experiment: 4L_2H_K16
Proposed as a replacement for the frozen Track A model: False

The selected configuration does not beat the Track A reference beyond noise (-0.0033 nats, 95% interval [-0.0048, -0.0020], seeds separated: False). The Track A model is retained. The selected configuration is reported as the best configuration found in this experiment, not as a replacement.


**RESULT.** 4L_2H_K16 (tuned setting) minus the Track A reference (2L_2H_K16, weight decay 0.1): −0.0033 nats on validation, 95% match-level bootstrap interval [−0.0048, −0.0020]. The seeds are **not** completely separated. The selected configuration does not beat the Track A reference beyond noise. **The Track A model is retained**; the selected configuration is reported as the best configuration found in this experiment, not as a replacement.

**Why the two decisions are kept apart.** "Best configuration in this experiment" is a ranking over the configurations tested here and always has an answer. "Replacement for the frozen Track A model" is a claim that the current model is measurably worse, and it needs evidence that satisfies both parts of the decision rule. Here the bootstrap interval excludes zero but the per-seed results overlap, so the second claim is not supported.

**Note.** The comparison changes two things at once: the architecture (4 against 2 layers) and the weight decay (0.01 against 0.1).

## 6a. Roles of the models in this experiment

Five questions are answered separately, each with its own evidence. No composite score is used.

1. **Frozen Track A reference** — the existing model (2 layers, weight decay 0.1).
2. **Best predictive configuration** — the benchmark candidate with the lowest mean validation NLL.
3. **Configurations within noise of the best** — the benchmark candidates the best one does not beat beyond noise.
4. **Smallest adequate configuration** — step 2 of the rule. This is the configuration frozen for final training and for the later mechanistic analysis.
5. **Replacement of the frozen Track A model** — step 4 of the rule.

**RESULT.** (1) Track A reference; (2) 8L_2H_K16; (3) 6L_1H_K16, 6L_2H_K16, 4L_2H_K16; beaten beyond noise by the best: 6L_8H_K16, 2L_2H_K16; (4) **4L_2H_K16**; (5) not supported on validation. Selection used validation data only.

In [ ]:
benchmark_versus = best_versus[best_versus["role"] == "benchmark"]
ROLES = {
    "frozen_track_a_reference": TRACK_A_CANDIDATE,
    "best_predictive_configuration": BEST_CANDIDATE,
    "within_noise_of_best": list(benchmark_versus[benchmark_versus["status"].str.startswith("within noise")]
                                 ["candidate"]),
    "beaten_beyond_noise_by_best": list(benchmark_versus[benchmark_versus["status"] == "beaten beyond noise by the best"]
                                        ["candidate"]),
    "smallest_adequate_configuration_frozen": SELECTED_CANDIDATE,
    "replaces_track_a": REPLACES_TRACK_A,
}
roles_table = pd.DataFrame([
    {"question": "Frozen Track A reference", "answer": ROLES["frozen_track_a_reference"]},
    {"question": "Best predictive configuration (lowest mean validation NLL)", "answer": ROLES["best_predictive_configuration"]},
    {"question": "Configurations within noise of the best", "answer": ", ".join(ROLES["within_noise_of_best"]) or "none"},
    {"question": "Configurations beaten beyond noise by the best", "answer": ", ".join(ROLES["beaten_beyond_noise_by_best"]) or "none"},
    {"question": "Smallest adequate configuration (frozen for notebook 13)", "answer": ROLES["smallest_adequate_configuration_frozen"]},
    {"question": "Replacement of the frozen Track A model supported on validation", "answer": str(ROLES["replaces_track_a"])},
])
print("Model selection is based on validation data only; test data were not used.")
print(roles_table.to_string(index=False))

Model selection is based on validation data only; test data were not used.
                                                       question                          answer
                                       Frozen Track A reference               Track A reference
     Best predictive configuration (lowest mean validation NLL)                       8L_2H_K16
                        Configurations within noise of the best 6L_1H_K16, 6L_2H_K16, 4L_2H_K16
                 Configurations beaten beyond noise by the best            6L_8H_K16, 2L_2H_K16
       Smallest adequate configuration (frozen for notebook 13)                       4L_2H_K16
Replacement of the frozen Track A model supported on validation                           False


## 7. The frozen configuration



In [ ]:
selected_candidate_record = [item for item in CANDIDATES if item["name"] == SELECTED_CANDIDATE][0]
FINAL_SEEDS = [100, 101, 102, 103, 104]      # the Track A final-training seeds

example_config = transformer_config(FINAL_SEEDS[0], selected_candidate_record["n_layers"],
                                    selected_candidate_record["n_heads"],
                                    selected_candidate_record["context_length"],
                                    d_model=selected_candidate_record["d_model"],
                                    d_ff=selected_candidate_record["d_ff"],
                                    learning_rate=selected_candidate_record["learning_rate"],
                                    weight_decay=selected_candidate_record["weight_decay"])
counted = model_module.count_parameters(
    training.build_model(example_config, bundle_for(example_config["context_length"])["vocabulary_sizes"]))

frozen_config = {
    "experiment": "hyperparameter_optimisation",
    "encoding": ENCODING,
    "possession_features": POSSESSION_FEATURES,
    "context_length": selected_candidate_record["context_length"],
    "stride": STRIDE,
    "model": {"n_layers": selected_candidate_record["n_layers"],
              "n_heads": selected_candidate_record["n_heads"],
              "d_model": selected_candidate_record["d_model"],
              "d_ff": selected_candidate_record["d_ff"]},
    "training": {"learning_rate": selected_candidate_record["learning_rate"],
                 "weight_decay": selected_candidate_record["weight_decay"],
                 "max_epochs": EPOCH_BUDGET,
                 "batch_size": selected_candidate_record["batch_size"],
                 "warmup_fraction": selected_candidate_record["warmup_fraction"],
                 "min_lr_ratio": training.DEFAULT_TRAINING["min_lr_ratio"],
                 "grad_clip": training.DEFAULT_TRAINING["grad_clip"],
                 "patience": training.DEFAULT_TRAINING["patience"]},
    "seed_policy": {"screening_seeds": [0, 1], "fine_search_seeds": [0, 1],
                    "stability_seeds": stability_report["seeds"], "final_seeds": FINAL_SEEDS,
                    "primary_checkpoint": "median validation NLL among the final seeds, "
                                          "as in Track A notebook 06"},
    "parameter_counts": counted,
    "validation_metrics_over_stability_seeds": {
        "n_seeds": int(selected_row["n_seeds"]),
        "nll_mean": float(selected_row["val_nll_mean"]), "nll_sd": float(selected_row["val_nll_sd"]),
        "accuracy_mean": float(selected_row["val_accuracy_mean"]),
        "macro_f1_mean": float(selected_row["val_macro_f1_mean"]),
        "top3_mean": float(selected_row["val_top3_mean"]), "top5_mean": float(selected_row["val_top5_mean"]),
        "ece_mean": float(selected_row["val_ece_mean"])},
    "selection": {
        "primary_metric": "mean validation NLL over the stability seeds",
        "candidate_set": "benchmark candidates: notebook 08 shortlist trained with the notebook 09 setting",
        "best_candidate": BEST_CANDIDATE,
        "selected_candidate": SELECTED_CANDIDATE,
        "rule": "smallest candidate the best one does not beat beyond noise; ordered by parameters, then "
                "head-position units, then mean validation NLL; 'beyond noise' = 95% match-level bootstrap "
                "interval below zero and complete seed separation (Track A notebook 04)",
        "replaces_track_a": REPLACES_TRACK_A,
        "replacement_statement": replacement_statement,
        "replacement_comparison": replacement_comparison,
        "saturation_point_supported": (elbow_report.get("saturation_point_supported")
                                       if elbow_report else None),
    },
    "roles": ROLES,
    "dataset_sha256": TRACK_A_FROZEN["dataset_sha256"],
    "track_a_frozen_config": TRACK_A_FROZEN,
    "run_mode": RUN_MODE,
    "frozen_at": pd.Timestamp.now(tz="UTC").isoformat(),
}

frozen_path = CONFIG_DIR / "frozen_config.json"
if frozen_path.exists():
    existing = json.load(open(frozen_path))
    comparable = ["encoding", "possession_features", "context_length", "model", "training"]
    if any(existing[key] != frozen_config[key] for key in comparable):
        raise RuntimeError("BLOCKED: a different configuration is already frozen for this experiment. "
                           "It is not overwritten. Delete it deliberately if the experiment is being redone.")
    print("A matching configuration is already frozen; it is kept unchanged.")
else:
    training.save_json(frozen_path, frozen_config)
    print("Configuration frozen:", frozen_path)

training.save_json(RESULTS_DIR / "12_selected_config.json", frozen_config)
print(json.dumps({key: frozen_config[key] for key in
                  ["context_length", "model", "training", "parameter_counts", "selection", "roles"]},
                 indent=2, default=str))

Configuration frozen: /content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/config/frozen_config.json
{
  "context_length": 16,
  "model": {
    "n_layers": 4,
    "n_heads": 2,
    "d_model": 128,
    "d_ff": 512
  },
  "training": {
    "learning_rate": 0.0003,
    "weight_decay": 0.01,
    "max_epochs": 30,
    "batch_size": 256,
    "warmup_fraction": 0.05,
    "min_lr_ratio": 0.1,
    "grad_clip": 1.0,
    "patience": 5
  },
  "parameter_counts": {
    "embeddings (tables + projections + positions)": 115840,
    "transformer blocks": 793088,
    "final LayerNorm": 256,
    "unembedding": 1935,
    "total": 911119,
    "fixed zero rows (never trained)": 512,
    "trainable": 910607
  },
  "selection": {
    "primary_metric": "mean validation NLL over the stability seeds",
    "candidate_set": "benchmark candidates: notebook 08 shortlist trained with the notebook 09 setting",
    "best_candidate": "8L_2H_K16",
    "selected_candidate": "4L_2H_K16",
    "ru

In [ ]:
check = {
    "selected_candidate": SELECTED_CANDIDATE,
    "selected_architecture": str(selected_row["architecture"]),
    "parameters": int(counted["total"]),
    "trainable_parameters": int(counted["trainable"]),
    "best_candidate_by_mean_nll": BEST_CANDIDATE,
    "selected_equals_best": SELECTED_CANDIDATE == BEST_CANDIDATE,
    "within_noise_of_best": ROLES["within_noise_of_best"],
    "replaces_track_a": REPLACES_TRACK_A,
    "frozen_config_written": str(frozen_path),
    "test_split_touched": False,
}
training.save_json(RESULTS_DIR / "12_summary.json", check)
print(json.dumps(check, indent=2))
print("\nThe test split remains locked. It is unlocked for the first time in notebook 13, "
      "and every unlock is appended to logs/test_access_log.jsonl.")

{
  "selected_candidate": "4L_2H_K16",
  "selected_architecture": "L4-H2-K16-D128-F512",
  "parameters": 911119,
  "trainable_parameters": 910607,
  "best_candidate_by_mean_nll": "8L_2H_K16",
  "selected_equals_best": false,
  "within_noise_of_best": [
    "6L_1H_K16",
    "6L_2H_K16",
    "4L_2H_K16"
  ],
  "replaces_track_a": false,
  "frozen_config_written": "/content/drive/MyDrive/soccer_mi/track_a/experiments/hyperparameter_optimisation/config/frozen_config.json",
  "test_split_touched": false
}

The test split remains locked. It is unlocked for the first time in notebook 13, and every unlock is appended to logs/test_access_log.jsonl.


## 8. Summary

| Item | Result |
|---|---|
| Candidate set | six benchmark architectures at the tuned setting (weight decay 0.01), seeds 10–14 |
| Best by mean validation NLL | 8L_2H_K16, 0.9010 |
| Within noise of the best | 6L_1H_K16, 6L_2H_K16, 4L_2H_K16 |
| **Selected (smallest within noise of the best)** | **4L_2H_K16**, 0.9036 (seed SD 0.0017), 911,119 parameters |
| Secondary metrics | 4th of 6 on NLL, accuracy, macro-F1, top-3, top-5; 5th on ECE |
| Selected − Track A reference (validation) | −0.0033 [−0.0048, −0.0020], seeds not separated |
| Replaces the Track A model | no — Track A model retained |
| Test split | not accessed |

`config/frozen_config.json` holds the selected configuration and the evidence behind it. It is the file that permits the test split to be unlocked for the final evaluation.
